In [1]:
import pandas as pd
import numpy as np
import os

RAW = "../data/raw"
PROCESSED = "../data/processed"

In [2]:
owid = pd.read_csv(f"{RAW}/owid_covid_data.csv", parse_dates=["date"])
print(owid.columns.tolist())


['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients', 'icu_patients_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests', 'total_tests_per_thousand', 'new_tests_per_thousand', 'new_tests_smoothed', 'new_tests_smoothed_per_thousand', 'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_boosters', 'new_vaccinations', 'new_vaccinations_smoothed', 'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred', 'total_

In [3]:
print("Loading JHU confirmed cases...")
jhu = pd.read_csv(f"{RAW}/jhu_confirmed_global.csv")

jhu_long = jhu.melt(
    id_vars=["Province/State", "Country/Region", "Lat", "Long"],
    var_name="date",
    value_name="confirmed"
)
jhu_long["date"] = pd.to_datetime(jhu_long["date"], format="%m/%d/%y")

jhu_country = (
    jhu_long
    .groupby(["Country/Region", "date"], as_index=False)["confirmed"]
    .sum()
    .rename(columns={"Country/Region": "country"})
)

jhu_country = jhu_country.sort_values(["country", "date"])
jhu_country["new_cases"] = (
    jhu_country.groupby("country")["confirmed"].diff().clip(lower=0)
)
jhu_country["new_cases_7day"] = (
    jhu_country.groupby("country")["new_cases"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

print(f"Shape: {jhu_country.shape} | Countries: {jhu_country['country'].nunique()}")
jhu_country.head()

Loading JHU confirmed cases...
Shape: (229743, 5) | Countries: 201


,country,date,confirmed,new_cases,new_cases_7day
0,Afghanistan,2020-01-22,0,NaN,NaN
1,Afghanistan,2020-01-23,0,0.0,0.0
2,Afghanistan,2020-01-24,0,0.0,0.0
3,Afghanistan,2020-01-25,0,0.0,0.0
4,Afghanistan,2020-01-26,0,0.0,0.0


In [4]:
print("Loading JHU deaths...")
jhu_d = pd.read_csv(f"{RAW}/jhu_deaths_global.csv")

jhu_d_long = jhu_d.melt(
    id_vars=["Province/State", "Country/Region", "Lat", "Long"],
    var_name="date",
    value_name="deaths"
)
jhu_d_long["date"] = pd.to_datetime(jhu_d_long["date"], format="%m/%d/%y")

jhu_deaths = (
    jhu_d_long
    .groupby(["Country/Region", "date"], as_index=False)["deaths"]
    .sum()
    .rename(columns={"Country/Region": "country"})
)
jhu_deaths["new_deaths"] = (
    jhu_deaths.groupby("country")["deaths"].diff().clip(lower=0)
)

print(f"Shape: {jhu_deaths.shape}")
jhu_deaths.head()

Loading JHU deaths...
Shape: (229743, 4)


,country,date,deaths,new_deaths
0,Afghanistan,2020-01-22,0,NaN
1,Afghanistan,2020-01-23,0,0.0
2,Afghanistan,2020-01-24,0,0.0
3,Afghanistan,2020-01-25,0,0.0
4,Afghanistan,2020-01-26,0,0.0


In [5]:
jhu_merged = jhu_country.merge(
    jhu_deaths[["country", "date", "deaths", "new_deaths"]],
    on=["country", "date"],
    how="left"
)

print(f"Merged shape: {jhu_merged.shape}")
jhu_merged.head()

Merged shape: (229743, 7)


,country,date,confirmed,new_cases,new_cases_7day,deaths,new_deaths
0,Afghanistan,2020-01-22,0,NaN,NaN,0,NaN
1,Afghanistan,2020-01-23,0,0.0,0.0,0,0.0
2,Afghanistan,2020-01-24,0,0.0,0.0,0,0.0
3,Afghanistan,2020-01-25,0,0.0,0.0,0,0.0
4,Afghanistan,2020-01-26,0,0.0,0.0,0,0.0


In [6]:
owid_cols = [
    "iso_code", "location", "date",
    "reproduction_rate",
    "total_vaccinations_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_tests_per_thousand",
    "positive_rate",
    "hosp_patients_per_million",
    "stringency_index",
    "population",
    "population_density",
    "median_age",
    "gdp_per_capita",
    "human_development_index",
    "hospital_beds_per_thousand"
]

owid_subset = owid[owid_cols].rename(columns={"location": "country"})
owid_subset = owid_subset[owid_subset["iso_code"].notna()]

# Remove continent-level aggregates (they have iso_code starting with OWID_)
owid_subset = owid_subset[~owid_subset["iso_code"].str.startswith("OWID_")]

print(f"OWID subset shape: {owid_subset.shape}")
owid_subset.head()

OWID subset shape: (395311, 16)


,iso_code,country,date,reproduction_rate,total_vaccinations_per_hundred,people_fully_vaccinated_per_hundred,total_tests_per_thousand,positive_rate,hosp_patients_per_million,stringency_index,population,population_density,median_age,gdp_per_capita,human_development_index,hospital_beds_per_thousand
0,AFG,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
1,AFG,Afghanistan,2020-01-06,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
2,AFG,Afghanistan,2020-01-07,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
3,AFG,Afghanistan,2020-01-08,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
4,AFG,Afghanistan,2020-01-09,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5


In [7]:
owid_cols = [
    "iso_code", "location", "date",
    "reproduction_rate",
    "total_vaccinations_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_tests_per_thousand",
    "positive_rate",
    "hosp_patients_per_million",
    "stringency_index",
    "population",
    "population_density",
    "median_age",
    "gdp_per_capita",
    "human_development_index",
    "hospital_beds_per_thousand"
]

owid_subset = owid[owid_cols].rename(columns={"location": "country"})
owid_subset = owid_subset[owid_subset["iso_code"].notna()]

# Remove continent-level aggregates (they have iso_code starting with OWID_)
owid_subset = owid_subset[~owid_subset["iso_code"].str.startswith("OWID_")]

print(f"OWID subset shape: {owid_subset.shape}")
owid_subset.head()

OWID subset shape: (395311, 16)


,iso_code,country,date,reproduction_rate,total_vaccinations_per_hundred,people_fully_vaccinated_per_hundred,total_tests_per_thousand,positive_rate,hosp_patients_per_million,stringency_index,population,population_density,median_age,gdp_per_capita,human_development_index,hospital_beds_per_thousand
0,AFG,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
1,AFG,Afghanistan,2020-01-06,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
2,AFG,Afghanistan,2020-01-07,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
3,AFG,Afghanistan,2020-01-08,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5
4,AFG,Afghanistan,2020-01-09,NaN,NaN,NaN,NaN,NaN,NaN,0.0,41128772,54.42,18.6,1803.99,0.51,0.5


In [8]:
df = jhu_merged.merge(owid_subset, on=["country", "date"], how="left")

# Forward fill demographic + vaccination columns per country
fill_cols = [
    "population", "population_density", "median_age",
    "gdp_per_capita", "human_development_index",
    "hospital_beds_per_thousand", "stringency_index",
    "total_vaccinations_per_hundred", "people_fully_vaccinated_per_hundred",
    "reproduction_rate"
]
df[fill_cols] = df.groupby("country")[fill_cols].ffill()

# Feature engineering
df["cases_per_million"] = (df["new_cases_7day"] / df["population"]) * 1_000_000
df["death_rate"] = df["deaths"] / df["confirmed"].replace(0, np.nan)

# Drop sparse early period
df = df[df["date"] >= "2020-03-01"].reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")
df[["country", "date", "new_cases", "new_cases_7day", "confirmed",
    "deaths", "cases_per_million", "reproduction_rate"]].head(10)

Final dataset shape: (221904, 23)


,country,date,new_cases,new_cases_7day,confirmed,deaths,cases_per_million,reproduction_rate
0,Afghanistan,2020-03-01,0.0,0.714286,5,0,0.017367,NaN
1,Afghanistan,2020-03-02,0.0,0.000000,5,0,0.000000,NaN
2,Afghanistan,2020-03-03,0.0,0.000000,5,0,0.000000,NaN
3,Afghanistan,2020-03-04,0.0,0.000000,5,0,0.000000,NaN
4,Afghanistan,2020-03-05,0.0,0.000000,5,0,0.000000,NaN
5,Afghanistan,2020-03-06,0.0,0.000000,5,0,0.000000,NaN
6,Afghanistan,2020-03-07,3.0,0.428571,8,0,0.010420,NaN
7,Afghanistan,2020-03-08,0.0,0.428571,8,0,0.010420,NaN
8,Afghanistan,2020-03-09,0.0,0.428571,8,0,0.010420,NaN
9,Afghanistan,2020-03-10,0.0,0.428571,8,0,0.010420,NaN


In [9]:
df.to_csv(f"{PROCESSED}/covid_merged.csv", index=False)
print(f"Saved to {PROCESSED}/covid_merged.csv")

# Quick summary
print(f"\nDate range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Countries : {df['country'].nunique()}")
print(f"Columns   : {df.shape[1]}")
print(f"\nMissing values (%):")
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False).head(10))

Saved to ../data/processed/covid_merged.csv

Date range: 2020-03-01 → 2023-03-09
Countries : 201
Columns   : 23

Missing values (%):
hosp_patients_per_million              86.5
total_tests_per_thousand               67.4
positive_rate                          60.6
people_fully_vaccinated_per_hundred    44.1
total_vaccinations_per_hundred         39.3
stringency_index                       18.4
hospital_beds_per_thousand             17.9
reproduction_rate                      16.3
median_age                             13.9
gdp_per_capita                         11.9
dtype: float64
